# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a guided template for loading and exploring a dataset using the `mlcroissant` library. All dataset entities are referenced using their `@id` fields, which uniquely identify each record set, field, and column in the Croissant schema.

### Dataset Source
The dataset is described by a Croissant schema available at the following URL:

https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and prepare to access records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Display dataset name and description metadata
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
List available record sets, their `@id` values, and the fields (columns) they contain using their `@id`s. This information is necessary to extract data in the next steps.


In [ ]:
# List all record sets and their fields using their `@id`
record_sets = dataset.record_sets
print("Available Record Sets and Their Fields:")
record_set_ids = []  # To be used next
for rs in record_sets:
    print(f"\n- RecordSet @id: {rs.id}")
    record_set_ids.append(rs.id)
    print("  Fields (columns):")
    for field in rs.fields:
        col_ids = []
        if hasattr(field, 'columns'):
            # field.columns is a list of Column objects
            for col in field.columns:
                col_ids.append(col.id)
        print(f"    - Field @id: {field.id} (columns @id: {', '.join(col_ids) if col_ids else 'N/A'})")

## 3. Data Extraction
Extract data by loading records from a specific record set using its `@id`. Each record is parsed as a dictionary with field (column) `@id`s as keys.


In [ ]:
# Prepare to extract all record sets available
# The variable `record_set_ids` was collected in the previous cell
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records, with columns (field @id): {list(df.columns)}")
        display(df.head())
    else:
        print("No records found for this record set.")

## 4. Exploratory Data Analysis (EDA)
Apply basic data exploration and processing steps:
- Filter records on a numeric field
- Normalize numeric data
- Group or aggregate records by a categorical field

All fields are referenced by their `@id`s. Please consult the record set data above for choosing relevant field `@id`s.


In [ ]:
# Example EDA: Filtering, normalizing, grouping on data from the main survey record set
# For illustration, let's select the first loaded record set. Adjust field @id as per available columns.
example_rs_id = record_set_ids[0] if record_set_ids else None
if example_rs_id is not None and example_rs_id in dataframes:
    df = dataframes[example_rs_id]
    print(f"Selected RecordSet @id: {example_rs_id}")
    print(f"Available field @id columns: {list(df.columns)}")
    # Example: suppose 'cr:GAD7_score' is a numeric field @id for GAD-7 assessment (adjust as appropriate)
    # If you know the actual field, change 'cr:GAD7_score' accordingly.
    possible_numeric_fields = [col for col in df.columns if df[col].dtype in [int, float, 'int64', 'float64', 'int32', 'float32']]
    if not possible_numeric_fields:
        # Try to infer numeric columns (some may be object type, so test conversion)
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except Exception:
                pass
        possible_numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print(f"Numeric fields detected by @id: {possible_numeric_fields}")
    if possible_numeric_fields:
        numeric_field = possible_numeric_fields[0]  # Select first numeric field
        print(f"Using numeric field for filtering and normalization: {numeric_field}")
        # Filter: keep records with a value above a threshold (e.g., 10)
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Records with {numeric_field} > {threshold}: {len(filtered_df)}")
        display(filtered_df.head())
        # Normalize the numeric field (z-score)
        filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized values for {numeric_field} (first 5 rows):")
        display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())
        # Try grouping by a categorical field (choose first non-numeric field)
        possible_categorical = [col for col in df.columns if col not in possible_numeric_fields]
        group_field = possible_categorical[0] if possible_categorical else None
        if group_field and group_field in filtered_df.columns:
            print(f"Grouping by field @id: {group_field}")
            grouped = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Mean {numeric_field} grouped by {group_field} (first 5 rows):")
            display(grouped.head())
        else:
            print("No categorical field available for grouping.")
    else:
        print("No numeric field detected for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields by referencing the field (column) `@id`s. For demonstration, we plot the normalized numeric field distribution (from the previous EDA), grouped by a categorical field if available.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of normalized scores by group (if available, from above)
if 'filtered_df' in locals() and not filtered_df.empty:
    if numeric_field + '_normalized' in filtered_df.columns:
        # If grouping field available, plot group distributions
        if group_field and group_field in filtered_df.columns:
            plt.figure(figsize=(8, 5))
            sns.boxplot(x=group_field, y=numeric_field + '_normalized', data=filtered_df)
            plt.title(f"Distribution of normalized {numeric_field} by {group_field}")
            plt.xlabel(group_field)
            plt.ylabel(f"{numeric_field} (normalized)")
            plt.show()
        else:
            plt.figure(figsize=(7, 4))
            sns.histplot(filtered_df[numeric_field + '_normalized'], kde=True)
            plt.title(f"Histogram of normalized {numeric_field}")
            plt.xlabel(f"{numeric_field} (normalized)")
            plt.ylabel("Frequency")
            plt.show()
    else:
        print("No normalized numeric field available for plotting.")
else:
    print("No filtered data available for visualization.")

## 6. Conclusion
- The dataset provides rich survey records on mental health in Kilifi County, Kenya, including demographic and psychological assessment information, referenced by standards-compliant `@id` identifiers.
- Using `mlcroissant`, we loaded the Croissant schema, examined available record sets and fields, and extracted tabular data using explicit `@id` references.
- Basic exploratory analysis and normalization could be performed by referencing these `@id`s, supporting reproducible AI-ready workflows.

### Next Steps
Further work could include statistical analysis of key assessment domains, advanced feature engineering using additional fields, and integration with ML pipelines, always referencing data entities by their `@id`s as best practice for FAIR data.
